# 第一题：使用sklearn的GaussianNB、BernoulliNB、MultinomialNB完成spambase垃圾邮件分类任务

实验内容：
1. 使用GaussianNB、BernoulliNB、MultinomialNB完成spambase邮件分类
2. 计算各自十折交叉验证的精度、查准率、查全率、F1值
3. 根据精度、查准率、查全率、F1值的实际意义以及四个值的对比阐述三个算法在spambase中的表现对比

# 1. 读取数据集

In [2]:
import numpy as np
from pathlib import Path

# 兼容从项目根目录或 solution 目录启动内核。
# 数据只从 data 目录读取，训练过程中不修改原始文件。
data_path = next(
    (base / "data" / "spambase" / "spambase.data"
     for base in (Path.cwd(), Path.cwd().parent)
     if (base / "data" / "spambase" / "spambase.data").is_file()),
    None,
)
if data_path is None:
    raise FileNotFoundError("请从项目根目录或 solution 目录运行本 Notebook。")
spambase = np.loadtxt(data_path, delimiter=",")
spamx = spambase[:, :57]
spamy = spambase[:, 57].astype(int)
assert spamx.shape == (4601, 57)
assert set(np.unique(spamy)) == {0, 1}
assert np.isfinite(spamx).all() and (spamx >= 0).all()
print(f"Spambase: {spamx.shape[0]} 个样本，{spamx.shape[1]} 个特征")
print(f"正常邮件: {(spamy == 0).sum()}；垃圾邮件: {(spamy == 1).sum()}")


Spambase: 4601 个样本，57 个特征
正常邮件: 2788；垃圾邮件: 1813


# 2. 导入模型

In [3]:
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
)


# 3. 计算十折交叉验证下，GaussianNB、BernoulliNB、MultinomialNB的精度、查准率、查全率、F1值

In [5]:
# 所有模型共用同一组分层十折划分，便于公平比较。
# 固定随机种子；每次只在九折训练，在剩余一折评价。
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
models = {
    "GaussianNB": GaussianNB(),
    "MultinomialNB": MultinomialNB(alpha=1.0),
    # BernoulliNB 将特征按是否大于 0 二值化，同时考虑出现和未出现。
    "BernoulliNB": BernoulliNB(alpha=1.0, binarize=0.0),
}
metric_names = ["accuracy", "precision", "recall", "f1"]
fold_records = []
oof_predictions = {}
for name, estimator in models.items():
    predictions = np.empty_like(spamy)
    for fold, (train_index, valid_index) in enumerate(cv.split(spamx, spamy), 1):
        model = clone(estimator)
        model.fit(spamx[train_index], spamy[train_index])
        predicted = model.predict(spamx[valid_index])
        predictions[valid_index] = predicted
        truth = spamy[valid_index]
        # 垃圾邮件 1 为正类；zero_division=0 避免无正类预测时分母为零。
        fold_records.append({
            "model": name, "fold": fold,
            "accuracy": accuracy_score(truth, predicted),
            "precision": precision_score(truth, predicted, zero_division=0),
            "recall": recall_score(truth, predicted, zero_division=0),
            "f1": f1_score(truth, predicted, zero_division=0),
        })
    oof_predictions[name] = predictions

fold_results = pd.DataFrame(fold_records)
cv_mean = fold_results.groupby("model", sort=False)[metric_names].mean()
cv_std = fold_results.groupby("model", sort=False)[metric_names].std(ddof=1)
print("分层十折交叉验证：各折指标的算术平均值")
display(cv_mean.round(6))
print("各折指标的样本标准差")
display(cv_std.round(6))

# 将每个样本恰好一次的折外预测汇总。此处的混淆矩阵不含训练集预测。
confusion_records = []
for name, predicted in oof_predictions.items():
    tn, fp, fn, tp = confusion_matrix(spamy, predicted, labels=[0, 1]).ravel()
    confusion_records.append({"model": name, "TN": tn, "FP": fp, "FN": fn, "TP": tp})
display(pd.DataFrame(confusion_records).set_index("model"))
assert len(fold_records) == 30


分层十折交叉验证：各折指标的算术平均值


,accuracy,precision,recall,f1
model,,,,
GaussianNB,0.820256,0.698906,0.957534,0.807845
MultinomialNB,0.790261,0.740695,0.721480,0.730591
BernoulliNB,0.885676,0.885503,0.815785,0.849037


各折指标的样本标准差


,accuracy,precision,recall,f1
model,,,,
GaussianNB,0.016408,0.021075,0.008623,0.014186
MultinomialNB,0.031884,0.045817,0.041791,0.040205
BernoulliNB,0.013258,0.023295,0.019896,0.017234


,TN,FP,FN,TP
model,,,,
GaussianNB,2038,750,77,1736
MultinomialNB,2328,460,505,1308
BernoulliNB,2596,192,334,1479


### 十折交叉验证结果

下表填写实际执行得到的十折均值。精度在本实验指 accuracy（准确率）；
查准率、查全率、F1 均以垃圾邮件 `1` 为正类。
标准差和折外预测混淆矩阵见上方输出。

| 算法 | 准确率 accuracy | 查准率 precision | 查全率 recall | F1 |
| --- | --- | --- | --- | --- |
| GaussianNB | 0.820256 | 0.698906 | 0.957534 | 0.807845 |
| MultinomialNB | 0.790261 | 0.740695 | 0.721480 | 0.730591 |
| BernoulliNB | 0.885676 | 0.885503 | 0.815785 | 0.849037 |


### 结果分析

Accuracy 表示全部邮件中分类正确的比例；precision 表示被判为垃圾邮件的邮件中，
确实是垃圾邮件的比例；recall 表示实际垃圾邮件被检出的比例；
F1 是 precision 和 recall 的调和平均数。
正常邮件误判为垃圾邮件记为 FP，垃圾邮件漏检记为 FN。

本次十折实验中，BernoulliNB 的平均准确率最高，BernoulliNB 的平均 F1 最高。

GaussianNB 的查全率为 0.957534，查准率为 0.698906；其折外预测共漏检 77 封垃圾邮件，并将 750 封正常邮件误判为垃圾邮件。它能保留连续特征大小的信息，但词频等特征包含大量零值、分布偏态，且特征之间存在相关性，高斯分布和条件独立假设只能近似成立。

MultinomialNB 的平均准确率为 0.790261、F1 为 0.730591。该模型通常用于非负计数；本数据混合了词频百分比、字符频率和大写字母长度统计，并非标准词袋计数，且各特征量纲差异明显，这可能影响多项式似然的适用性。

BernoulliNB 的平均准确率为 0.885676、F1 为 0.849037。按是否大于零二值化后，它利用词或字符出现及未出现的证据；这一表示舍弃频率幅度，也降低了不同量纲带来的影响。上述结论限于当前数据、默认模型设置和指定划分。十折表中的 precision、recall、F1 为逐折指标均值，不是将折外混淆矩阵合并后再计算的值。
